# INCEpTION gold corpus - annotator comment inventory (G)

This is a **qualitative review whose product is a list of pending work**. It is not
a measurement and must never be cited as performance evidence. The quantitative
evaluation in `E_evaluation.ipynb` deliberately excludes interpretation of comment
text, and nothing produced here feeds back into it.

Notebook G describes the comment population without grouping or categorising it.


## G1. Perimeter

Load both validated span tables and retain only rows whose
`comment_status == "annotator_note"`.


In [1]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 360)

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / "data/gcn_gold_corpus").is_dir())
CORPUS = ROOT / "data/gcn_gold_corpus"
OUTPUT = ROOT / "data/interim/gcn_gold_comments"
OUTPUT.mkdir(parents=True, exist_ok=True)

frames = []
for layer, filename, category_column in [
    ("evidence", "evidence_spans.parquet", "label"),
    ("photometry", "photometry_spans.parquet", "measurement_type"),
]:
    frame = pd.read_parquet(CORPUS / filename)
    frame = frame.loc[frame["comment_status"].eq("annotator_note")].copy()
    frame["layer"] = layer
    frame["label_or_measurement_type"] = frame[category_column]
    frame["annotator"] = frame["layer_source"]
    frames.append(frame)
comments = pd.concat(frames, ignore_index=True, sort=False)
if comments.empty:
    raise ValueError("No annotator_note comments were loaded")
annotator_names = sorted(pd.read_parquet(
    CORPUS / "annotators.parquet", columns=["annotator"])["annotator"].dropna().unique())

g1 = pd.DataFrame([
    {"measure": "all annotator notes", "count": len(comments)},
    {"measure": "evidence", "count": comments["layer"].eq("evidence").sum()},
    {"measure": "photometry", "count": comments["layer"].eq("photometry").sum()},
])
display(g1)
print(f"Input directory: {CORPUS}")


,measure,count
0,all annotator notes,894
1,evidence,293
2,photometry,601


Input directory: /home/meneses/project_astronomical/MAFORAI/data/gcn_gold_corpus


## G2. Distribution by annotator, layer, and document

Shares use all annotator-note rows as the denominator. Layer-specific leaders are
also printed so overall concentration does not conceal different review workloads.


In [2]:
comment_counts = (comments.groupby(["annotator", "layer"]).size()
                  .unstack(fill_value=0).reset_index())
by_annotator = pd.DataFrame({"annotator": annotator_names}).merge(
    comment_counts, on="annotator", how="left")
for layer in ["evidence", "photometry"]:
    if layer not in by_annotator:
        by_annotator[layer] = 0
    by_annotator[layer] = by_annotator[layer].fillna(0).astype(int)
by_annotator["total"] = by_annotator["evidence"] + by_annotator["photometry"]
by_annotator["share_pct"] = 100 * by_annotator["total"] / len(comments)
by_annotator = by_annotator.sort_values(
    ["total", "annotator"], ascending=[False, True]).reset_index(drop=True)

by_document = (comments.groupby(["document_name", "layer"]).size()
               .unstack(fill_value=0).reset_index())
for layer in ["evidence", "photometry"]:
    if layer not in by_document:
        by_document[layer] = 0
by_document["total"] = by_document["evidence"] + by_document["photometry"]
by_document = by_document.sort_values(
    ["total", "document_name"], ascending=[False, True]).reset_index(drop=True)

display(by_annotator)
display(by_document)
leader = by_annotator.iloc[0]
print(f"Overall leader: {leader['annotator']} - {leader['total']} comments "
      f"({leader['share_pct']:.2f}%).")
for layer in ["evidence", "photometry"]:
    layer_counts = comments.loc[comments["layer"].eq(layer), "annotator"].value_counts()
    print(f"{layer} leader: {layer_counts.index[0]} - {layer_counts.iloc[0]} of "
          f"{layer_counts.sum()} ({100 * layer_counts.iloc[0] / layer_counts.sum():.2f}%).")


,annotator,evidence,photometry,total,share_pct
0,Sarah,53,286,339,37.919463
1,Dahlia,51,143,194,21.700224
2,Priyadarshini,82,101,183,20.469799
3,Zhanat,38,27,65,7.270694
4,Camille,30,8,38,4.250559
5,Patrice,13,17,30,3.355705
6,Eslam,22,4,26,2.908277
7,Yodgor,4,15,19,2.125280
8,Andrii,0,0,0,0.000000
9,Xinyue,0,0,0,0.000000


layer,document_name,evidence,photometry,total
0,event_GRB241030.xmi,97,147,244
1,event_GCN-251013_173943.xmi,31,182,213
2,event_2026owq.xmi,25,98,123
3,event_GCN-260604_202037.xmi,14,68,82
4,event_GCN-251222_170549.xmi,36,38,74
5,event_GRB-260708A.xmi,48,8,56
6,event_EP-260623_025405.xmi,14,29,43
7,event_2025aji.xmi,11,19,30
8,event_GCN-260614_134953.xmi,11,7,18
9,event_GRB-241025_013651.xmi,6,5,11


Overall leader: Sarah - 339 comments (37.92%).
evidence leader: Priyadarshini - 82 of 293 (27.99%).
photometry leader: Sarah - 286 of 601 (47.59%).


## G3. Comment-only versus accompanying a content correction

A row is **comment-only** only when parsed `changed_fields` equals `['comment']`
exactly. The comment-only subset is where a rejection or standalone remark can
live; comments with other changed fields usually explain a correction already
visible in structured data. This describes the two subsets without interpreting
individual comments.


In [3]:
def parse_changed_fields(value):
    if not isinstance(value, str) or not value:
        return []
    parsed = json.loads(value)
    if not isinstance(parsed, list):
        raise TypeError(f"changed_fields is not a list: {value!r}")
    return parsed


comments["parsed_changed_fields"] = comments["changed_fields"].map(parse_changed_fields)
comments["comment_subset"] = np.where(
    comments["parsed_changed_fields"].map(lambda fields: fields == ["comment"]),
    "comment_only", "with_content_change")
g3 = (comments.groupby(["layer", "comment_subset"]).size()
      .unstack(fill_value=0).reindex(["evidence", "photometry"]).reset_index())
g3["total"] = g3.select_dtypes(include="number").sum(axis=1)
display(g3)


comment_subset,layer,comment_only,with_content_change,total
0,evidence,156,137,293
1,photometry,274,327,601


## G4. Structural context of the comments

The following tables cross comments with `match_status`, the existing layer
category, exact `changed_fields` combinations, and individual changed fields.
These are corpus columns, not qualitative groups.


In [4]:
g4_status = (comments.groupby(["layer", "match_status"]).size()
             .reset_index(name="comments")
             .sort_values(["layer", "comments"], ascending=[True, False]))
g4_categories = (comments.assign(
    category=comments["label_or_measurement_type"].fillna("<UNSET>"))
    .groupby(["layer", "category"]).size().reset_index(name="comments")
    .sort_values(["layer", "comments", "category"], ascending=[True, False, True]))
g4_combinations = (comments.groupby(["layer", "changed_fields"], dropna=False).size()
                   .reset_index(name="comments")
                   .sort_values(["layer", "comments"], ascending=[True, False])
                   .groupby("layer", group_keys=False).head(15))
exploded_fields = (comments[["layer", "parsed_changed_fields"]]
                   .explode("parsed_changed_fields")
                   .rename(columns={"parsed_changed_fields": "changed_field"}))
exploded_fields["changed_field"] = exploded_fields["changed_field"].fillna("<NONE>")
g4_fields = (exploded_fields.groupby(["layer", "changed_field"]).size()
             .reset_index(name="comments")
             .sort_values(["layer", "comments", "changed_field"],
                          ascending=[True, False, True]))
display(g4_status)
display(g4_categories)
display(g4_combinations)
display(g4_fields)


,layer,match_status,comments
0,evidence,corrected,194
1,evidence,created,99
2,photometry,corrected,513
3,photometry,created,88


,layer,category,comments
13,evidence,TRIGGER_INSTRUMENT,96
7,evidence,LOCALIZATION,51
14,evidence,TRIGGER_TIME,33
10,evidence,REDSHIFT_EVENT,26
2,evidence,COUNTERPART_ASSOCIATION,19
4,evidence,HIGH_ENERGY_PROPERTY,14
3,evidence,EVENT_IDENTITY,13
0,evidence,<UNSET>,12
6,evidence,LIGHTCURVE_EVOLUTION,9
11,evidence,SPECTROSCOPY,5


,layer,changed_fields,comments
6,evidence,"[""comment""]",156
7,evidence,[],99
0,evidence,"[""certainty"", ""comment""]",14
5,evidence,"[""comment"", ""value""]",13
3,evidence,"[""comment"", ""target""]",5
1,evidence,"[""comment"", ""label"", ""target""]",3
4,evidence,"[""comment"", ""unit""]",2
2,evidence,"[""comment"", ""label""]",1
50,photometry,"[""comment""]",274
51,photometry,[],88


,layer,changed_field,comments
2,evidence,comment,194
0,evidence,<NONE>,99
1,evidence,certainty,14
6,evidence,value,13
4,evidence,target,8
3,evidence,label,4
5,evidence,unit,2
9,photometry,comment,513
11,photometry,instrument,143
15,photometry,obs_time_reference,107


## G5. Text inventory

Text length is measured on the verbatim comment. For recurrence, normalisation is
limited to Unicode-aware case folding, stripping the ends, and collapsing internal
whitespace. No semantic interpretation, stemming, or synonym merging is applied.


In [5]:
comments["normalised_text"] = comments["comment"].fillna("").map(
    lambda value: re.sub(r"\s+", " ", value.strip().casefold()))
lengths = comments["comment"].fillna("").str.len()
g5_lengths = pd.DataFrame([{
    "n": len(lengths), "min": int(lengths.min()),
    "p25": lengths.quantile(0.25), "median": lengths.median(),
    "p75": lengths.quantile(0.75), "p90": lengths.quantile(0.90),
    "p95": lengths.quantile(0.95), "max": int(lengths.max()),
}])
text_counts = comments["normalised_text"].value_counts()
g5_vocabulary = pd.DataFrame([{
    "distinct_normalised_texts": len(text_counts),
    "singleton_texts": int(text_counts.eq(1).sum()),
    "comments_in_singletons": int(text_counts.loc[text_counts.eq(1)].sum()),
}])
frequent = (comments.groupby("normalised_text", as_index=False)
            .agg(count=("normalised_text", "size"),
                 distinct_annotators=("annotator", "nunique"),
                 representative_text=("comment", "first"))
            .sort_values(["count", "normalised_text"], ascending=[False, True])
            .head(40)[["representative_text", "count", "distinct_annotators"]]
            .reset_index(drop=True))
display(g5_lengths)
display(g5_vocabulary)
display(frequent)
print(f"Top-40 texts used by one annotator: "
      f"{int(frequent['distinct_annotators'].eq(1).sum())} of {len(frequent)}")


,n,min,p25,median,p75,p90,p95,max
0,894,2,19.0,40.0,62.0,108.7,137.0,454


,distinct_normalised_texts,singleton_texts,comments_in_singletons
0,451,346,346


,representative_text,count,distinct_annotators
0,ignore amateur,53,1
1,AB,46,2
2,we ignore master,32,1
3,not taking into account by LLM,27,1
4,Primary trigger instrument - first alert,18,1
5,photometric system is unknown. changed to observation_start,17,1
6,Added instrument name,14,1
7,Fermi/GBM,11,1
8,changed absolute_time to observation_mid,9,1
9,how did we know this is vega? Added inst as UVOT. time is relative to trigger but should be indicated that its T-start.,9,1


Top-40 texts used by one annotator: 39 of 40


## G6. Reading export

Export every annotator-note row with its structural context and verbatim text.


In [6]:
export_columns = [
    "document_name", "layer_source", "layer", "match_status", "changed_fields",
    "label_or_measurement_type", "begin", "end", "covered_text", "comment",
]
export = comments[export_columns].sort_values(
    ["document_name", "layer", "layer_source", "begin", "end"],
    kind="stable").reset_index(drop=True)
export_path = OUTPUT / "all_annotator_comments.csv"
export.to_csv(export_path, index=False, lineterminator="\n")
print(f"Wrote {len(export)} rows to {export_path}")
display(export.head(5))


Wrote 894 rows to /home/meneses/project_astronomical/MAFORAI/data/interim/gcn_gold_comments/all_annotator_comments.csv


,document_name,layer_source,layer,match_status,changed_fields,label_or_measurement_type,begin,end,covered_text,comment
0,event_2025aji.xmi,Camille,evidence,created,[],LOCALIZATION,2291,2314,"RA, Dec 198.689, +5.039",Uncertaincy of 3 arcmin
1,event_2025aji.xmi,Camille,evidence,created,[],LOCALIZATION,2754,2780,"RA, Dec 198.67654,\n5.03019",position of an xray source; uncertaincy 1.9 arcsec
2,event_2025aji.xmi,Camille,evidence,corrected,"[""comment""]",REDSHIFT_EVENT,13728,13745,redshift z = 2.15,redshift context
3,event_2025aji.xmi,Camille,evidence,created,[],LOCALIZATION,21894,21923,"RA, Dec = 198.67662, +5.03073",with uncertaincy of 2.4arcsec
4,event_2025aji.xmi,Camille,evidence,corrected,"[""comment""]",REDSHIFT_EVENT,23274,23290,redshift of 2.15,already reported in a previous circular


## G7. Closing observation

The inventory contains both repeated stock phrases and many one-off formulations;
recurrence is often concentrated within one annotator rather than shared across
reviewers. Comment-only remarks and comments accompanying structured corrections
are both substantial subsets. Therefore notebook H must publish explicit criteria,
apply them in a fixed precedence order, expose annotator/document support, and leave
unmatched comments as residue rather than treating the vocabulary as controlled.

This remains a qualitative preparation of pending work, not performance evidence.
